<a href="https://colab.research.google.com/github/kunapareddysriram2007-cmyk/Deep_Learning/blob/main/Repo_git_pytorch_cuda_setup_colab_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning — Google Colab Setup (Safe Version)

This notebook is designed for a **fresh Google Colab runtime**. It clones/reuses the configured GitHub repository, checks the environment, installs only the repository dependencies when available, verifies PyTorch/CUDA, and performs safe source-code checks.

> **Important:** Colab runs on a temporary Linux machine. Your local RTX 4050 is not available inside Colab. To use a Colab GPU, select **Runtime → Change runtime type → GPU**. PyTorch recommends checking `torch.cuda.is_available()` after selecting a CUDA-enabled runtime. citeturn955933search0


In [ ]:
import os
import sys
import platform
import shutil
import subprocess
from pathlib import Path

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
REMOTE_URL = "https://github.com/kunapareddysrir2007-cmyk/Deep_Learning.git"
REPO_NAME = "Deep_Learning"
PROJECT_DIR = BASE_DIR / REPO_NAME

print("Python       :", sys.version.split()[0])
print("Executable   :", sys.executable)
print("OS           :", platform.platform())
print("Working dir  :", Path.cwd())
print("Git          :", shutil.which("git") or "not installed")
print("nvidia-smi   :", shutil.which("nvidia-smi") or "not available")


Python       : 3.13.18
Executable   : /usr/bin/python3
OS           : Linux-6.6.159+-x86_64-with-glibc2.39
Working dir  : /content
Git          : /usr/bin/git
nvidia-smi   : not available


In [ ]:
def run_cmd(args, cwd=None, timeout=120):
    """Run a command without crashing the notebook; return CompletedProcess or None."""
    args = [str(x) for x in args]
    print("\n$", " ".join(args))
    try:
        r = subprocess.run(args, cwd=cwd, text=True, capture_output=True, timeout=timeout)
    except FileNotFoundError:
        print("Command not found:", args[0])
        return None
    except subprocess.TimeoutExpired:
        print("Command timed out.")
        return None
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.stderr.strip():
        print(r.stderr.strip())
    print("Exit code:", r.returncode)
    return r

run_cmd(["git", "--version"])
run_cmd(["nvidia-smi"], timeout=30)



$ git --version
git version 2.43.0
Exit code: 0

$ nvidia-smi
Command not found: nvidia-smi


## 1. Clone or reuse the repository

The original notebook hard-coded the repository URL above. This version does **not** assume that `/content/VisionPort` exists. It derives the project path from the configured repository name.

If the repository is private and GitHub asks for authentication, the notebook will report that cloning was unsuccessful instead of producing an uncontrolled traceback.


In [ ]:
def is_git_repo(path):
    return (path / ".git").exists()

REPO_READY = False

if PROJECT_DIR.exists() and is_git_repo(PROJECT_DIR):
    print("Existing Git repository found:", PROJECT_DIR)
    REPO_READY = True
    # Only update a clean repository. Never overwrite uncommitted work.
    status = run_cmd(["git", "status", "--porcelain"], cwd=PROJECT_DIR)
    if status is not None and status.returncode == 0 and not status.stdout.strip():
        print("Repository is clean; attempting a fast-forward update.")
        run_cmd(["git", "pull", "--ff-only"], cwd=PROJECT_DIR, timeout=120)
    else:
        print("Repository has uncommitted changes or status could not be read; skipping pull.")
elif PROJECT_DIR.exists():
    print(f"{PROJECT_DIR} exists but is not a Git repository.")
    print("Choose a different folder or remove/rename the existing folder before cloning.")
else:
    print("Cloning:", REMOTE_URL)
    result = run_cmd(["git", "clone", "--depth", "1", REMOTE_URL, PROJECT_DIR], timeout=180)
    REPO_READY = result is not None and result.returncode == 0 and is_git_repo(PROJECT_DIR)

if REPO_READY:
    os.chdir(PROJECT_DIR)
    print("\nRepository ready:", Path.cwd())
else:
    print("\nRepository is not available in this runtime.")
    print("The remaining environment checks can still run; repository-specific checks will be skipped.")


Cloning: https://github.com/kunapareddysrir2007-cmyk/Deep_Learning.git

$ git clone --depth 1 https://github.com/kunapareddysrir2007-cmyk/Deep_Learning.git /content/Deep_Learning
Cloning into '/content/Deep_Learning'...
fatal: could not read Username for 'https://github.com': No such device or address
Exit code: 128

Repository is not available in this runtime.
The remaining environment checks can still run; repository-specific checks will be skipped.


In [ ]:
if REPO_READY:
    print("Repository contents (top level):")
    for p in sorted(Path(".").iterdir(), key=lambda x: x.name.lower()):
        print(" -", p.name + ("/" if p.is_dir() else ""))

    for name in ["requirements.txt", "requirements-colab.txt", "pyproject.toml", "setup.py", "environment.yml", "README.md"]:
        p = Path(name)
        if p.exists():
            print(f"Found {name} ({p.stat().st_size} bytes)")


## 2. Install repository dependencies safely

This cell uses the **same Python interpreter as the notebook kernel** (`sys.executable -m pip`). It installs `requirements-colab.txt` if the repository provides one; otherwise it uses `requirements.txt` if present. If neither exists, nothing is installed.

For PyTorch, avoid blindly replacing the Colab-provided build unless the project actually requires a different version. PyTorch's Colab guidance notes that Colab may already have a suitable PyTorch installation and recommends checking the installed versions first. citeturn955933search0


In [ ]:
if REPO_READY:
    requirement_file = None
    for candidate in ["requirements-colab.txt", "requirements.txt"]:
        if Path(candidate).exists():
            requirement_file = candidate
            break

    if requirement_file:
        print("Installing dependencies from", requirement_file)
        result = run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", requirement_file], timeout=600)
        if result is None or result.returncode != 0:
            print("Dependency installation reported a problem. Review the output above.")
    else:
        print("No requirements file found; skipping pip installation.")
else:
    print("Repository unavailable; skipping repository dependency installation.")


Repository unavailable; skipping repository dependency installation.


## 3. PyTorch and CUDA check

This is intentionally device-agnostic: it works on CPU-only Colab runtimes and CUDA runtimes. If CUDA is unavailable, it reports that fact instead of calling CUDA-only functions.


In [ ]:
try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("PyTorch CUDA build:", torch.version.cuda)
    cuda_ok = bool(torch.cuda.is_available())
    print("CUDA available:", cuda_ok)
    print("GPU count:", torch.cuda.device_count())
    if cuda_ok:
        for i in range(torch.cuda.device_count()):
            print(f"GPU {i}:", torch.cuda.get_device_name(i))
        DEVICE = torch.device("cuda")
    else:
        DEVICE = torch.device("cpu")
        print("Using CPU because CUDA is unavailable.")
except Exception as exc:
    print("PyTorch check failed:", type(exc).__name__, str(exc))
    DEVICE = None


PyTorch version: 2.14.1+cpu
PyTorch CUDA build: None
CUDA available: False
GPU count: 0
Using CPU because CUDA is unavailable.


In [ ]:
if DEVICE is not None:
    try:
        x = torch.randn((1024, 1024), device=DEVICE)
        y = torch.randn((1024, 1024), device=DEVICE)
        z = x @ y
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        print("Tensor device:", z.device)
        print("Tensor shape :", tuple(z.shape))
        print("Finite result:", bool(torch.isfinite(z).all().item()))
        del x, y, z
    except Exception as exc:
        print("Tensor test failed:", type(exc).__name__, str(exc))
else:
    print("Tensor test skipped because PyTorch was not initialized.")


Tensor device: cpu
Tensor shape : (1024, 1024)
Finite result: True


In [ ]:
if DEVICE is not None and DEVICE.type == "cuda":
    try:
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        print(f"Free GPU memory : {free_bytes / 1024**3:.2f} GiB")
        print(f"Total GPU memory: {total_bytes / 1024**3:.2f} GiB")
        print(f"Allocated       : {torch.cuda.memory_allocated() / 1024**2:.1f} MiB")
        print(f"Reserved        : {torch.cuda.memory_reserved() / 1024**2:.1f} MiB")
    except Exception as exc:
        print("GPU memory report failed:", type(exc).__name__, str(exc))
else:
    print("GPU memory report skipped: CUDA is unavailable.")


GPU memory report skipped: CUDA is unavailable.


## 4. Static checks for the actual repository

When the repository cloned successfully, this section checks Python source files for syntax errors and reports common hard-coded local-path patterns. It does not modify project source files.


In [ ]:
if REPO_READY:
    python_files = list(Path(".").rglob("*.py"))
    print("Python files found:", len(python_files))
    syntax_errors = []
    for p in python_files:
        # Skip virtual environments and common generated folders.
        if any(part in {".git", ".venv", "venv", "site-packages", "__pycache__"} for part in p.parts):
            continue
        try:
            compile(p.read_text(encoding="utf-8", errors="replace"), str(p), "exec")
        except SyntaxError as exc:
            syntax_errors.append((str(p), exc.lineno, exc.msg))

    if syntax_errors:
        print("Syntax errors found:")
        for item in syntax_errors:
            print(" -", item)
    else:
        print("No Python syntax errors found in scanned source files.")

    suspicious = []
    patterns = ["C:\\\\", "C:/Users/", "\\\\Users\\", "/Users/", "/home/"]
    for p in python_files:
        if any(part in {".git", ".venv", "venv", "site-packages", "__pycache__"} for part in p.parts):
            continue
        try:
            text = p.read_text(encoding="utf-8", errors="replace")
        except Exception:
            continue
        if any(pattern in text for pattern in patterns):
            suspicious.append(str(p))
    print("Files containing possible local-machine paths:", suspicious if suspicious else "none detected")
else:
    print("Repository unavailable; static source checks skipped.")


Repository unavailable; static source checks skipped.


## 5. Git status — read-only

These cells do not commit or push anything. That prevents accidental changes to your GitHub repository.


In [ ]:
if REPO_READY:
    run_cmd(["git", "status", "--short"], cwd=PROJECT_DIR)
    run_cmd(["git", "branch", "--show-current"], cwd=PROJECT_DIR)
    run_cmd(["git", "remote", "-v"], cwd=PROJECT_DIR)
else:
    print("Git status skipped because the repository is unavailable.")


Git status skipped because the repository is unavailable.


## 6. Optional Git commit/push

GitHub authentication is deliberately **not automated** here. A failed login, missing permission, protected branch, or private repository can legitimately prevent a push. Do not put a GitHub password or token directly into notebook source.

If you want to push later, use GitHub CLI, SSH, or a configured credential manager.


In [ ]:
PUSH_CHANGES = False

if PUSH_CHANGES and REPO_READY:
    print("Push was explicitly enabled, but this safe notebook does not store GitHub credentials.")
    print("Authenticate with GitHub CLI/SSH/credential manager first, then run:")
    print("git add -A")
    print("git commit -m 'Update project'")
    print("git push")
else:
    print("No Git push performed.")


No Git push performed.


## Final result

A successful run should show:

- Repository ready **or** a clear GitHub access/network message (without a traceback).
- PyTorch version.
- `CUDA available: True` when a GPU runtime is assigned; otherwise CPU fallback.
- A successful tensor test.
- No Python syntax errors in scanned repository files when the repo is accessible.

For a GPU runtime, enable the GPU under **Runtime → Change runtime type** before running the notebook. PyTorch's documentation recommends this setup for CUDA-enabled Colab tutorials. citeturn955933search0
